# Bootstrap

In [40]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np

## Get data

### Bootstrap

In [41]:
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/bootstrap.sql'
parameters = {
    'start_date': '2026-04-01',
}

bqc = BigQueryConnector()
# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 22.42 GB when run.
Estimated query cost: $0.15


In [42]:
data_bootstrap = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache
refresh_data = True
if refresh_data:
    data_bootstrap = bqc.get(query='./sql/bootstrap.sql', is_path=True, query_parameters=parameters)
    data_bootstrap.to_pickle('./data/bootstrap.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data_bootstrap = pd.read_pickle('./data/bootstrap.pkl')

: 

In [26]:
data_bootstrap

,session_id,payload_timestamp,payload_date,user_id,tutorial_step_id,context,seconds,is_first_session,sequence
0,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.782472Z,2026-04-02,10003F6FA8F5957E,app_launch,cmpt,0.355,False,1
1,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.783575Z,2026-04-02,10003F6FA8F5957E,display_loading_screen,strt,0.389,False,2
2,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.784469Z,2026-04-02,10003F6FA8F5957E,service_manager_process,strt,0.406,False,3
3,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.785349Z,2026-04-02,10003F6FA8F5957E,service_manager_process,cmpt,0.798,False,4
4,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.786193Z,2026-04-02,10003F6FA8F5957E,auth_process,strt,0.806,False,5
...,...,...,...,...,...,...,...,...,...
156989015,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.558424Z,2026-04-17,FFFFF67B4ED7017A,helpshift_sdk,strt,15.502,True,3
156989016,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.560164Z,2026-04-17,FFFFF67B4ED7017A,helpshift_sdk,cmpt,15.504,True,4
156989017,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:45.940106Z,2026-04-17,FFFFF67B4ED7017A,external_notifications_popup,strt,20.883,True,5
156989018,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:46.152842Z,2026-04-17,FFFFF67B4ED7017A,external_notifications_popup,cmpt,21.096,True,6


# Bootstrap event analysis

_Section in progress._ This section will identify the earliest in-game milestone that best predicts long-term retention — the "bootstrap moment" — by comparing event completion rates between retained and churned players.

In [27]:
data_bootstrap = data_bootstrap[(data_bootstrap.context == 'cmpt')&(data_bootstrap.payload_date < pd.to_datetime('today'))]
data_bootstrap.loc[data_bootstrap.seconds > 600, 'seconds'] = 601

data_bootstrap = data_bootstrap[data_bootstrap['payload_date'] >= pd.to_datetime('2026-04-03')]

data_bootstrap

,session_id,payload_timestamp,payload_date,user_id,tutorial_step_id,context,seconds,is_first_session,sequence
18,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:02.630413Z,2026-04-06,10003F6FA8F5957E,app_launch,cmpt,0.285,False,1
21,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:02.633224Z,2026-04-06,10003F6FA8F5957E,service_manager_process,cmpt,0.682,False,4
23,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:06.078734Z,2026-04-06,10003F6FA8F5957E,auth_process,cmpt,4.175,False,6
25,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:06.083439Z,2026-04-06,10003F6FA8F5957E,helpshift_sdk,cmpt,4.180,False,8
27,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:30.614235Z,2026-04-06,10003F6FA8F5957E,display_assets_download_popup,cmpt,28.711,False,10
...,...,...,...,...,...,...,...,...,...
156989012,None,2026-04-17T15:45:39.192703Z,2026-04-17,FFFFF67B4ED7017A,service_manager_process,cmpt,14.127,True,186440
156989014,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.555821Z,2026-04-17,FFFFF67B4ED7017A,auth_process,cmpt,15.499,True,2
156989016,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.560164Z,2026-04-17,FFFFF67B4ED7017A,helpshift_sdk,cmpt,15.504,True,4
156989018,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:46.152842Z,2026-04-17,FFFFF67B4ED7017A,external_notifications_popup,cmpt,21.096,True,6


In [35]:
# calculate global percentiles to remove high value outliers 
percentiles = data_bootstrap.groupby(['tutorial_step_id'])['seconds'].quantile([0.1, 0.5, 0.9]).unstack()
percentiles.columns = ['P10', 'P50', 'P90']
percentiles.reset_index(inplace=True)

# set tutorial_step_id order with a tuple of the tutorial_step_id in the desired order
tutorial_step_order = {'01':'app_launch', '02':'display_loading_screen', '03':'external_notifications_popup', '04':'display_applovin_consent', '05':'external_do_not_track_popup', '06':'display_terms_of_service', '07':'auth_process', '08':'helpshift_sdk', '09':'update_app_popup', '10':'service_manager_process', '11':'display_assets_download_popup'}
tutorial_step_order

# join step order to data_bootstrap
data_bootstrap = data_bootstrap.merge(pd.DataFrame(list(tutorial_step_order.items()), columns=['step_order', 'tutorial_step_id']), on='tutorial_step_id', how='left')

# create a non-outliers version of the data by filtering out rows where seconds is greater than the P90 for that tutorial_step_id
data_bootstrap_no_outliers = data_bootstrap.merge(percentiles[['tutorial_step_id', 'P90']], on='tutorial_step_id', how='left')
data_bootstrap_no_outliers = data_bootstrap_no_outliers[data_bootstrap_no_outliers['seconds'] < data_bootstrap_no_outliers['P90']]

# calculate the percentiles P90, P50 and P10 for the seconds metric by payload_date and tutorial_step_id
percentiles_by_day = data_bootstrap.groupby(['payload_date', 'tutorial_step_id'])['seconds'].quantile([0.1, 0.5, 0.9]).unstack()
percentiles_by_day.columns = ['P10', 'P50', 'P90']
percentiles_by_day.reset_index(inplace=True)

percentiles_by_day_melted = percentiles_by_day.melt(id_vars=['payload_date', 'tutorial_step_id'], var_name='percentile', value_name='seconds').sort_values(['payload_date', 'tutorial_step_id', 'percentile']).reset_index(drop=True)

percentiles

,tutorial_step_id,P10,P50,P90
0,app_launch,0.1940,0.275,0.5610
1,auth_process,1.3840,2.168,4.6930
2,display_applovin_consent,3.7860,7.437,25.7456
3,display_assets_download_popup,10.5050,28.971,145.8402
4,display_loading_screen,8.3140,14.381,69.3330
5,display_terms_of_service,7.5136,12.094,32.3118
6,external_do_not_track_popup,13.5774,26.905,69.4052
7,external_notifications_popup,12.3398,25.330,92.3668
8,helpshift_sdk,1.3900,2.191,4.8160
9,service_manager_process,0.4870,0.756,1.8310


In [36]:
data_bootstrap

,session_id,payload_timestamp,payload_date,user_id,tutorial_step_id,context,seconds,is_first_session,sequence,tutorial_step_name,step_order
0,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:02.630413Z,2026-04-06,10003F6FA8F5957E,app_launch,cmpt,0.285,False,1,NaN,01
1,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:02.633224Z,2026-04-06,10003F6FA8F5957E,service_manager_process,cmpt,0.682,False,4,NaN,10
2,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:06.078734Z,2026-04-06,10003F6FA8F5957E,auth_process,cmpt,4.175,False,6,NaN,07
3,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:06.083439Z,2026-04-06,10003F6FA8F5957E,helpshift_sdk,cmpt,4.180,False,8,NaN,08
4,7bfc0cab-97a7-405c-b086-9d7e0e323c02,2026-04-06T15:56:30.614235Z,2026-04-06,10003F6FA8F5957E,display_assets_download_popup,cmpt,28.711,False,10,NaN,11
...,...,...,...,...,...,...,...,...,...,...,...
83386860,None,2026-04-17T15:45:39.192703Z,2026-04-17,FFFFF67B4ED7017A,service_manager_process,cmpt,14.127,True,186440,NaN,10
83386861,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.555821Z,2026-04-17,FFFFF67B4ED7017A,auth_process,cmpt,15.499,True,2,NaN,07
83386862,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.560164Z,2026-04-17,FFFFF67B4ED7017A,helpshift_sdk,cmpt,15.504,True,4,NaN,08
83386863,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:46.152842Z,2026-04-17,FFFFF67B4ED7017A,external_notifications_popup,cmpt,21.096,True,6,NaN,03


In [39]:
data_bootstrap[data_bootstrap['user_id'] == '10003F6FA8F5957E'].sort_values('payload_timestamp').sort_values(['session_id','step_order'])

,session_id,payload_timestamp,payload_date,user_id,tutorial_step_id,context,seconds,is_first_session,sequence,tutorial_step_name,step_order
6,03727d2c-ee9a-4458-9cd7-3e4a5a9199f6,2026-04-06T16:06:12.631089Z,2026-04-06,10003F6FA8F5957E,app_launch,cmpt,0.214,False,1,NaN,01
10,03727d2c-ee9a-4458-9cd7-3e4a5a9199f6,2026-04-06T16:06:28.610956Z,2026-04-06,10003F6FA8F5957E,display_loading_screen,cmpt,16.587,False,9,NaN,02
8,03727d2c-ee9a-4458-9cd7-3e4a5a9199f6,2026-04-06T16:06:16.091935Z,2026-04-06,10003F6FA8F5957E,auth_process,cmpt,4.070,False,6,NaN,07
9,03727d2c-ee9a-4458-9cd7-3e4a5a9199f6,2026-04-06T16:06:16.095248Z,2026-04-06,10003F6FA8F5957E,helpshift_sdk,cmpt,4.073,False,8,NaN,08
7,03727d2c-ee9a-4458-9cd7-3e4a5a9199f6,2026-04-06T16:06:12.633968Z,2026-04-06,10003F6FA8F5957E,service_manager_process,cmpt,0.573,False,4,NaN,10
...,...,...,...,...,...,...,...,...,...,...,...
31,f8497274-8369-457e-950e-00f33debbd5f,2026-04-11T16:16:02.830193Z,2026-04-11,10003F6FA8F5957E,app_launch,cmpt,0.251,False,1,NaN,01
35,f8497274-8369-457e-950e-00f33debbd5f,2026-04-11T16:16:20.206693Z,2026-04-11,10003F6FA8F5957E,display_loading_screen,cmpt,18.006,False,9,NaN,02
33,f8497274-8369-457e-950e-00f33debbd5f,2026-04-11T16:16:06.574051Z,2026-04-11,10003F6FA8F5957E,auth_process,cmpt,4.383,False,6,NaN,07
34,f8497274-8369-457e-950e-00f33debbd5f,2026-04-11T16:16:06.579436Z,2026-04-11,10003F6FA8F5957E,helpshift_sdk,cmpt,4.389,False,8,NaN,08


In [29]:
data_bootstrap_agg = data_bootstrap.groupby(['payload_date', 'tutorial_step_id']).agg(
    unique_users = ('user_id', 'nunique'),
    avg_seconds = ('seconds', 'mean'),
).reset_index()

data_bootstrap_agg

,payload_date,tutorial_step_id,unique_users,avg_seconds
0,2026-04-03,app_launch,93216,1.151598
1,2026-04-03,auth_process,92597,5.677143
2,2026-04-03,display_applovin_consent,1894,15.196625
3,2026-04-03,display_assets_download_popup,5468,85.506946
4,2026-04-03,display_loading_screen,91111,58.036128
...,...,...,...,...
517,2026-05-21,external_do_not_track_popup,61,41.546295
518,2026-05-21,external_notifications_popup,1406,43.668500
519,2026-05-21,helpshift_sdk,81626,5.167850
520,2026-05-21,service_manager_process,81863,1.901709


In [30]:
fig = px.line(data_bootstrap_agg, 
              x='payload_date', 
              y='unique_users',
              color='tutorial_step_id',
              title='Unique Users by Step Type and Payload Date',
              width=1200,
              height=600,
              hover_data={'unique_users': True, 'payload_date': True, 'tutorial_step_id': True},
)
fig.show()

In [31]:
fig = px.bar(data_bootstrap_agg, 
              x='payload_date', 
              y='avg_seconds',
              color='tutorial_step_id',
              title='Average Seconds by Step Type and Payload Date',
              width=1200,
              height=600,
              hover_data={'unique_users': True, 'avg_seconds': True, 'payload_date': True, 'tutorial_step_id': True},
)
fig.show()

In [32]:
fig = px.line(percentiles_by_day_melted[percentiles_by_day_melted['percentile'] == 'P50'], 
              x='payload_date', 
              y='seconds',
              color='tutorial_step_id',
              facet_row='percentile',
              title='Percentiles for step duration by step type and payload date',
              width=1200,
              height=600,
              hover_data={'seconds': True, 'payload_date': True, 'tutorial_step_id': True},
)
fig.show()

In [33]:
# Filter data for update_app_popup tutorial_step_id
update_app_popup_data = data_bootstrap[data_bootstrap['tutorial_step_id'] == 'update_app_popup']
display_assets_download_popup_data = data_bootstrap[data_bootstrap['tutorial_step_id'] == 'display_assets_download_popup']

chart_data = display_assets_download_popup_data

# Set threshold for minimum user representation
min_users_threshold = 1

# Filter update_app_popup_data to remove tutorial_step_ids with fewer than min_users_threshold unique users
chart_data_filtered = chart_data[
    chart_data.groupby('tutorial_step_id')['user_id'].transform('nunique') >= min_users_threshold
]

fig = px.histogram(chart_data_filtered, 
                   x='seconds',
                   nbins=100,
                   title='Distribution of Users by Seconds for x event',
                   labels={'seconds': 'Seconds', 'count': 'Number of Users'},
                   width=1200,
                   height=600,
                   log_x=False,)

# Add percentile lines
percentile_values = percentiles[percentiles['tutorial_step_id'] == 'update_app_popup'].iloc[0]
for percentile in ['P10', 'P50', 'P90']:
    fig.add_vline(x=percentile_values[percentile], 
                  line_dash="dash", 
                  line_color="red",
                  annotation_text=percentile,
                  annotation_position="top")

fig.show()